In [1]:
# 1. Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 2. Đặt thư mục Cache HuggingFace trỏ về Drive
os.environ["HF_HOME"] = "/content/drive/MyDrive/vllm_cache"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # Tránh cảnh báo sampler trên T4

# 3. Cài đặt thư viện
!pip install -q "vllm==0.29.0" huggingface_hub

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [11]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


In [2]:
from huggingface_hub import snapshot_download

draft_repo = "leptonai/EAGLE-Qwen2.5-7B-Instruct"

print(f"Đang tải draft checkpoint: {draft_repo}...")
draft_path = snapshot_download(
    repo_id=draft_repo,
    local_dir=f"/content/drive/MyDrive/vllm_cache/hub/{draft_repo.replace('/', '--')}"
)
print("Tải thành công tại:", draft_path)

Đang tải draft checkpoint: leptonai/EAGLE-Qwen2.5-7B-Instruct...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Tải thành công tại: /content/drive/MyDrive/vllm_cache/hub/leptonai--EAGLE-Qwen2.5-7B-Instruct


In [1]:
from vllm import LLM, SamplingParams

# Cấu hình speculative decoding
llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    enforce_eager=True,
    speculative_config={
        "method": "eagle",
        "model": "leptonai/EAGLE-Qwen2.5-7B-Instruct",
        "num_speculative_tokens": 3,  # Bắt đầu thử với k = 3 token nháp
    }
)

INFO 09-18 16:11:26 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'speculative_config': {'method': 'eagle', 'model': 'leptonai/EAGLE-Qwen2.5-7B-Instruct', 'num_speculative_tokens': 3}, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 09-18 16:12:03 [model.py:684] Resolved architecture: Qwen2ForCausalLM
INFO 09-18 16:12:03 [model.py:2021] Using max model len 2048


config.json: reconstructing file:   0%|          |  0.00B /   737B            

config.json: downloading bytes:           |  0.00B            

INFO 09-18 16:12:07 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-18 16:12:07 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-18 16:12:07 [model.py:2021] Using max model len 131072


ValidationError: 1 validation error for SpeculativeConfig
  Value error, Model architectures ['EagleQwen2ForCausalLM'] are not supported for now. Supported architectures: dict_keys(['AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AXK1ForCausalLM', 'BailingMoeForCausalLM', 'BailingMoeV2ForCausalLM', 'BailingMoeV2_5ForCausalLM', 'BailingMoeV3ForCausalLM', 'BloomForCausalLM', 'ChatGLMModel', 'ChatGLMForConditionalGeneration', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CwmForCausalLM', 'DbrxForCausalLM', 'DeciLMForCausalLM', 'DeepseekForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'ExaoneForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FalconH1ForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForCausalLM', 'Rnj1ForCausalLM', 'Gemma3nForCausalLM', 'Gemma4ForCausalLM

=> Trên vLLM hiện tại, không thể nạp trực tiếp checkpoint EAGLE dạng EagleQwen2ForCausalLM vì thiếu class đăng ký kiến trúc trong ModelRegistry => Dùng tạm 1 con model 0.5B cùng loại để nháp

# Dùng model nhỏ hơn cùng dòng để nháp

In [2]:
from vllm import LLM, SamplingParams

# Khởi tạo Speculative Decoding với draft model Qwen2.5-0.5B
llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    enforce_eager=True,
    speculative_config={
        "method": "draft_model",
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "num_speculative_tokens": 3,
    }
)

sampling_params = SamplingParams(temperature=0.0, max_tokens=256)
outputs = llm.generate(["Viết một đoạn văn ngắn giới thiệu về Vịnh Hạ Long:"], sampling_params)
print(outputs[0].outputs[0].text)


INFO 09-18 16:31:09 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'speculative_config': {'method': 'draft_model', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'num_speculative_tokens': 3}, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 09-18 16:31:09 [model.py:684] Resolved architecture: Qwen2ForCausalLM
INFO 09-18 16:31:09 [model.py:2021] Using max model len 2048


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

INFO 09-18 16:31:10 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-18 16:31:10 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-18 16:31:10 [model.py:2021] Using max model len 32768
INFO 09-18 16:31:10 [speculative.py:1586] Overriding draft model max model len from 32768 to 2048


ValidationError: 1 validation error for SpeculativeConfig
  Value error, Target and draft model should have the same vocabulary size. Target model vocab_size=152064. Draft model vocab_size=151936. Using models with different tokenizers can cause out-of-bounds errors during speculative decoding. [type=value_error, input_value=ArgsKwargs((), {'method':...very_timeout_sec=120))}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

## Xử lý validator của vLLM quá cứng nhắc khi ép target_vocab == draft_vocab dù cho 2 model 7B và 0.9B lệch nhau 128 token trong vocab_size

In [5]:
%%writefile test_spec.py
from vllm import LLM, SamplingParams

def main():
    print(">>> Đang khởi tạo vLLM với Speculative Decoding...")
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct-AWQ",
        quantization="awq",
        dtype="half",
        gpu_memory_utilization=0.85,
        max_model_len=2048,
        enforce_eager=True,
        speculative_config={
            "method": "draft_model",
            "model": "Qwen/Qwen2.5-0.5B-Instruct",
            "num_speculative_tokens": 3,
            "use_heterogeneous_vocab": True,
            "draft_sample_method": "greedy",
        }
    )

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=256
    )

    prompt = "Viết một bài văn ngắn khoảng 150 từ ca ngợi vẻ đẹp của Vịnh Hạ Long bằng tiếng Việt."
    outputs = llm.generate([prompt], sampling_params)

    for output in outputs:
        print("\n================ KẾT QUẢ SINH TOKEN ================")
        print(output.outputs[0].text)

if __name__ == "__main__":
    main()

Writing test_spec.py


In [6]:
!python test_spec.py

>>> Đang khởi tạo vLLM với Speculative Decoding...
INFO 09-18 16:39:28 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'speculative_config': {'method': 'draft_model', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'num_speculative_tokens': 3, 'use_heterogeneous_vocab': True, 'draft_sample_method': 'greedy'}, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 09-18 16:39:28 [model.py:684] Resolved architecture: Qwen2ForCausalLM
INFO 09-18 16:39:28 [model.py:2021] Using max model len 2048
INFO 09-18 16:39:31 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-18 16:39:31 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-18 16:39:31 [model.py:2021] Using max model len 32768
INFO 09-18 16:39:31 [speculative.py:1586] Overriding draft model max model len from 32768 to 2048
INFO 09-18 16:39:31 [scheduler.py:277] Chunked prefill is enabled with 

In [7]:
%%writefile test_spec1.py
import time
from vllm import LLM, SamplingParams

def main():
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct-AWQ",
        quantization="awq",
        dtype="half",
        gpu_memory_utilization=0.85,
        max_model_len=2048,
        enforce_eager=True,
        speculative_config={
            "method": "draft_model",
            "model": "Qwen/Qwen2.5-0.5B-Instruct",
            "num_speculative_tokens": 3,
            "use_heterogeneous_vocab": True,
            "draft_sample_method": "greedy",
        }
    )

    sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

    # 1. Warmup run (để Triton biên dịch xong toàn bộ kernel)
    print("\n>>> Đang chạy Warmup...")
    llm.generate(["Xin chào, bạn là ai?"], sampling_params)

    # 2. Benchmark run chính thức
    prompt = "Viết một bài văn ngắn khoảng 150 từ ca ngợi vẻ đẹp của Vịnh Hạ Long bằng tiếng Việt."
    print("\n>>> Đang chạy Benchmark chính thức (Warm Run)...")
    start_time = time.time()
    outputs = llm.generate([prompt], sampling_params)
    elapsed = time.time() - start_time

    text = outputs[0].outputs[0].text
    num_tokens = len(outputs[0].outputs[0].token_ids)
    tok_per_sec = num_tokens / elapsed

    print(f"\n[KẾT QUẢ ĐO ĐẠC]")
    print(f"- Số token sinh ra: {num_tokens}")
    print(f"- Thời gian: {elapsed:.2f} giây")
    print(f"- Tốc độ sinh chữ: {tok_per_sec:.2f} tok/s")
    print(f"\n[Văn bản sinh ra]:\n{text}")

if __name__ == "__main__":
    main()


Writing test_spec1.py


In [8]:
!python test_spec1.py

INFO 09-18 16:49:24 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'speculative_config': {'method': 'draft_model', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'num_speculative_tokens': 3, 'use_heterogeneous_vocab': True, 'draft_sample_method': 'greedy'}, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 09-18 16:49:25 [model.py:684] Resolved architecture: Qwen2ForCausalLM
INFO 09-18 16:49:25 [model.py:2021] Using max model len 2048
INFO 09-18 16:49:27 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-18 16:49:27 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-18 16:49:27 [model.py:2021] Using max model len 32768
INFO 09-18 16:49:27 [speculative.py:1586] Overriding draft model max model len from 32768 to 2048
INFO 09-18 16:49:27 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
Parse safetensors file